# Mixed Factor Model

This notebook builds linear z-score features, binned factors, ordered target encoding, and a LightGBM model.

In [ ]:
%pip install lightgbm

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import KBinsDiscretizer
from lightgbm import LGBMRegressor, log_evaluation

DATA_DIR = Path('D:/Kaggle/MLbook/stocks-return-prediction-v3')
TARGET_COL = 'y'

train = pd.read_pickle(DATA_DIR / 'train_data.pkl')
test = pd.read_pickle(DATA_DIR / 'test_data.pkl')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

train = train.sort_values(['date', 'code']).reset_index(drop=True)
feature_cols = sorted(
    [c for c in train.columns if c.startswith('f_')],
    key=lambda c: int(c.split('_')[1])
)

def add_zscores(frame):
    frame = frame.copy()
    means = frame.groupby('date')[feature_cols].transform('mean')
    stds = frame.groupby('date')[feature_cols].transform('std')
    zscores = (frame[feature_cols] - means) / (stds + 0.01)
    zscores.columns = [f'{c}_zscore' for c in feature_cols]
    return pd.concat([frame, zscores], axis=1)

train = add_zscores(train)
test = add_zscores(test)

linear_cols = [f'{c}_zscore' for c in feature_cols]
VALID_DAYS = 60
unique_dates = np.sort(train['date'].unique())
valid_dates = unique_dates[-VALID_DAYS:]
fit_mask = ~train['date'].isin(valid_dates)
print('Training rows:', fit_mask.sum())
print('Validation dates:', len(valid_dates))
bin_cols = []

for feature in feature_cols:
    n_bins = int(np.clip(
        train.loc[fit_mask, feature].nunique() // 50,
        40,
        1000
    ))

    discretizer = KBinsDiscretizer(
        n_bins=n_bins,
        encode='ordinal',
        strategy='quantile',
        random_state=2026
    )

    discretizer.fit(train.loc[fit_mask, [feature]])
    bin_col = f'{feature}_bin'
    train[bin_col] = discretizer.transform(train[[feature]]).ravel()
    test[bin_col] = discretizer.transform(test[[feature]]).ravel()
    bin_cols.append(bin_col)

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Linear feature count:', len(linear_cols))
print('Binned feature count:', len(bin_cols))

In [ ]:
class OrderedTE:
    def __init__(self, smoothing=1.0):
        self.smoothing = smoothing

    def fit(self, frame, category_cols, target_col):
        self.category_cols = list(category_cols)
        self.target_col = target_col
        self.global_prior_ = float(frame[target_col].mean())
        self.stats_ = {}
        result = frame.copy()

        for column in self.category_cols:
            temp = frame[[column, target_col]].copy()
            temp['_count_previous'] = temp.groupby(
                column, sort=False
            ).cumcount()
            temp['_sum_previous'] = (
                temp.groupby(column, sort=False)[target_col].cumsum()
                - temp[target_col]
            )
            result[f'{column}_TE'] = (
                temp['_sum_previous']
                + self.smoothing * self.global_prior_
            ) / (
                temp['_count_previous'] + self.smoothing
            )
            self.stats_[column] = (
                frame.groupby(column, dropna=False)[target_col]
                .agg(['count', 'mean'])
                .reset_index()
            )

        return result

    def transform(self, frame):
        result = frame.copy()

        for column in self.category_cols:
            count_col = f'{column}_count'
            mean_col = f'{column}_mean'
            statistics = self.stats_[column].rename(
                columns={'count': count_col, 'mean': mean_col}
            )
            result = result.merge(
                statistics,
                on=column,
                how='left',
                sort=False
            )
            result[f'{column}_TE'] = (
                result[count_col] * result[mean_col]
                + self.smoothing * self.global_prior_
            ) / (
                result[count_col] + self.smoothing
            )
            result[f'{column}_TE'] = result[
                f'{column}_TE'
            ].fillna(self.global_prior_)
            result.drop(
                columns=[count_col, mean_col],
                inplace=True
            )

        return result


category_cols = ['code'] + bin_cols
te_cols = [f'{c}_TE' for c in category_cols]
mixed_cols = linear_cols + bin_cols + te_cols

train_part = train.loc[fit_mask].copy()
valid_part = train.loc[~fit_mask].copy()

te = OrderedTE(smoothing=1.0)
train_encoded = te.fit(
    train_part,
    category_cols=category_cols,
    target_col=TARGET_COL
)
valid_encoded = te.transform(valid_part)

print('Mixed feature count:', len(mixed_cols))
print('Missing columns:', [c for c in mixed_cols if c not in train_encoded.columns])

In [ ]:
def mean_rank_ic(frame, predictions):
    evaluation = frame[['date', TARGET_COL]].copy()
    evaluation['prediction'] = predictions
    daily_ics = []

    for _, group in evaluation.groupby('date', sort=False):
        if group[TARGET_COL].nunique() < 2:
            continue
        if group['prediction'].nunique() < 2:
            continue
        ic = group[TARGET_COL].corr(
            group['prediction'],
            method='spearman'
        )
        if not pd.isna(ic):
            daily_ics.append(ic)

    return float(np.mean(daily_ics))

lgb_params = {
    'objective': 'l1',
    'boosting_type': 'gbdt',
    'num_leaves': 10,
    'max_depth': 4,
    'learning_rate': 0.002,
    'n_estimators': 256,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 50,
    'reg_lambda': 50,
    'random_state': 2026,
    'n_jobs': -1,
    'verbose': -1
}

model = LGBMRegressor(**lgb_params)
model.fit(
    train_encoded[mixed_cols],
    train_encoded[TARGET_COL],
    eval_X=valid_encoded[mixed_cols],
    eval_y=valid_encoded[TARGET_COL],
    eval_metric='l1',
    callbacks=[log_evaluation(10)]
)

valid_predictions = model.predict(valid_encoded[mixed_cols])
validation_ic = mean_rank_ic(valid_encoded, valid_predictions)
print('Mean Daily Rank IC:', validation_ic)

search_configs = [
    {
        'name': 'baseline',
        'num_leaves': 10,
        'max_depth': 4,
        'learning_rate': 0.002,
        'n_estimators': 256,
        'reg_alpha': 50,
        'reg_lambda': 50,
        'subsample_freq': 0
    },
    {
        'name': 'shallow',
        'num_leaves': 6,
        'max_depth': 3,
        'learning_rate': 0.002,
        'n_estimators': 256,
        'reg_alpha': 50,
        'reg_lambda': 50,
        'subsample_freq': 0
    },
    {
        'name': 'wider',
        'num_leaves': 15,
        'max_depth': 5,
        'learning_rate': 0.002,
        'n_estimators': 256,
        'reg_alpha': 50,
        'reg_lambda': 50,
        'subsample_freq': 0
    },
    {
        'name': 'more_trees',
        'num_leaves': 10,
        'max_depth': 4,
        'learning_rate': 0.001,
        'n_estimators': 512,
        'reg_alpha': 50,
        'reg_lambda': 50,
        'subsample_freq': 0
    },
    {
        'name': 'faster_learning',
        'num_leaves': 10,
        'max_depth': 4,
        'learning_rate': 0.005,
        'n_estimators': 128,
        'reg_alpha': 50,
        'reg_lambda': 50,
        'subsample_freq': 0
    },
    {
        'name': 'weaker_regularization',
        'num_leaves': 10,
        'max_depth': 4,
        'learning_rate': 0.002,
        'n_estimators': 256,
        'reg_alpha': 10,
        'reg_lambda': 10,
        'subsample_freq': 0
    },
    {
        'name': 'stronger_regularization',
        'num_leaves': 10,
        'max_depth': 4,
        'learning_rate': 0.002,
        'n_estimators': 256,
        'reg_alpha': 100,
        'reg_lambda': 100,
        'subsample_freq': 0
    },
    {
        'name': 'bagging',
        'num_leaves': 10,
        'max_depth': 4,
        'learning_rate': 0.002,
        'n_estimators': 256,
        'reg_alpha': 50,
        'reg_lambda': 50,
        'subsample_freq': 1
    }
]

search_results = []

for config in search_configs:
    name = config['name']
    params = {k: v for k, v in config.items() if k != 'name'}

    candidate = LGBMRegressor(
        objective='l1',
        boosting_type='gbdt',
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=2026,
        n_jobs=-1,
        verbose=-1,
        **params
    )

    candidate.fit(
        train_encoded[mixed_cols],
        train_encoded[TARGET_COL]
    )

    candidate_predictions = candidate.predict(
        valid_encoded[mixed_cols]
    )

    candidate_ic = mean_rank_ic(
        valid_encoded,
        candidate_predictions
    )

    search_results.append({
        'name': name,
        'rank_ic': candidate_ic,
        **params
    })

    print(f'{name}: Mean Daily Rank IC={candidate_ic:.6f}')

search_results_df = pd.DataFrame(search_results).sort_values(
    'rank_ic',
    ascending=False
).reset_index(drop=True)

print(search_results_df)

best_config = search_results_df.iloc[0].to_dict()
best_params = {
    key: best_config[key]
    for key in [
        'num_leaves',
        'max_depth',
        'learning_rate',
        'n_estimators',
        'reg_alpha',
        'reg_lambda',
        'subsample_freq'
    ]
}

lgb_params.update(best_params)
print('Best parameters:', lgb_params)

In [ ]:
te_full = OrderedTE(smoothing=1.0)
train_full_encoded = te_full.fit(
    train,
    category_cols=category_cols,
    target_col=TARGET_COL
)
test_full_encoded = te_full.transform(test)

model_full = LGBMRegressor(**lgb_params)
model_full.fit(
    train_full_encoded[mixed_cols],
    train_full_encoded[TARGET_COL],
    callbacks=[log_evaluation(10)]
)

test_predictions = model_full.predict(
    test_full_encoded[mixed_cols]
)

prediction_frame = test[['code', 'date']].copy()
prediction_frame['y_pred'] = test_predictions

submission = sample_submission.drop(
    columns=['y_pred'],
    errors='ignore'
).merge(
    prediction_frame,
    on=['code', 'date'],
    how='left',
    sort=False
)

assert len(submission) == len(sample_submission)
assert submission['y_pred'].notna().all()

submission_path = DATA_DIR / 'submission_mixed_lgbm.csv'
submission.to_csv(submission_path, index=False)

print('Saved:', submission_path)
print(submission.head())